In [15]:
import os
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from torch.utils.data import DataLoader
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sklearn.model_selection import train_test_split

from ThesisModelFunctionsOriginal import *

In [74]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### Load Data

In [37]:
# Retrieve all data and queries
corpus = retrieve_corpus()
queries, data = retrieve_training_data()

In [44]:
# Split train and test sets
train_data, test_data = train_test_split(list(data.items()), test_size=0.2, random_state=42)
train_data = dict(train_data)
test_data = dict(test_data)

In [77]:
# Load train data into Dataset
queries_text = []
rel_docs_text = []

for query_id, rel_docs_id in zip(list(train_data.keys()), list(train_data.values())):
    for rel_doc in rel_docs_id:
        queries_text.append(queries[query_id])
        rel_docs_text.append(corpus[rel_doc])

train_dataset = Dataset.from_dict({"anchor": queries_text, "positive": rel_docs_text})

In [78]:
# Load test data into Evaluator
val_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=test_data,
    #name="all-MiniLM-L6-v2-test"
)

### Load Model to Finetune

In [79]:
model = SentenceTransformer("all-MiniLM-L6-v2").to(device)

### Define a Loss Function

In [80]:
loss = MultipleNegativesRankingLoss(model)

### Specify training arguments

In [81]:
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="models/all-MiniLM-L6-v2",
    # Optional training parameters:
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    #warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
)

### Create a trainer & train

In [82]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    #eval_dataset=eval_dataset,
    loss=loss,
    evaluator=val_evaluator,
)
trainer.train()

Step,Training Loss


Step,Training Loss


TrainOutput(global_step=135, training_loss=0.7061729148582175, metrics={'train_runtime': 14.1307, 'train_samples_per_second': 296.164, 'train_steps_per_second': 9.554, 'total_flos': 0.0, 'train_loss': 0.7061729148582175, 'epoch': 4.851851851851852})

### Create an evaluator and evaluate the trained model on the test set

In [83]:
test_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=test_data,
    #name="all-MiniLM-L6-v2-test"
)
test_results = test_evaluator(model)

In [84]:
test_results

{'cosine_accuracy@1': 0.6296296296296297,
 'cosine_accuracy@3': 0.7654320987654321,
 'cosine_accuracy@5': 0.8209876543209876,
 'cosine_accuracy@10': 0.8888888888888888,
 'cosine_precision@1': 0.6296296296296297,
 'cosine_precision@3': 0.26543209876543206,
 'cosine_precision@5': 0.1802469135802469,
 'cosine_precision@10': 0.10185185185185183,
 'cosine_recall@1': 0.558641975308642,
 'cosine_recall@3': 0.6995884773662551,
 'cosine_recall@5': 0.7664609053497942,
 'cosine_recall@10': 0.8420781893004115,
 'cosine_ndcg@10': 0.7173023550837545,
 'cosine_mrr@10': 0.7098814422888498,
 'cosine_map@100': 0.6675114858724602}